# Tennis Data Cleaning Pipeline

This notebook performs comprehensive data cleaning and validation on ATP tennis match point-by-point (pbp) data.

## Overview
- Loads raw match data from CSV
- Validates match winners against scores
- Parses point-by-point play data
- Removes duplicates and invalid records
- Cleans temporal and duration fields
- Exports cleaned dataset

In [43]:

import pandas as pd

## Data Loading


In [44]:
df=pd.read_csv('/Users/ssingodia/Downloads/Tennis/Data/pbp_matches_atp_main_archive.csv')

## 2. Initial Data Exploration

In [45]:
df.head(10)

,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,adf_flag,wh_minutes
0,2231275,28 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Olivier Rochus,Fabio Fognini,2,SSSS;RRRR;SSRRSS;SSRRSS;RSRSRSRR;SSRSS;RSRRSR;...,6-4 6-1,0,66
1,2231276,28 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Robin Haase,Marin Cilic,2,SSRSS;RRSSRSSS;SSSS;RSSSS;SRSRSS;RSRSRSSS;RSRS...,4-6 6-4 6-3,0,141
2,2236280,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Marin Cilic,Andreas Seppi,1,SSSS;SRRRR;SSRRRSSS;RSRRSSSS;RSRSSS;SRRRR;SSRS...,6-1 6-3,0,71
3,2233792,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Alexandr Dolgopolov,Albert Ramos,1,SRSSRS;RRSSSRSS;RSRSSRSRSS;RRRR;SSRSS;SRSSS;SR...,6-3 7-5,0,83
4,2233791,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Carlos Berlocq,Juan Carlos Ferrero,2,RRRR;SRSSS;SSRRSS;RSRSSS;RRRSR;SSSRRS;RRRR.SRR...,6-1 7-6(5),0,150
5,2229262,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Matthias Bachinger,Julien Benneteau,2,SSSS;SSSS;RRRR;SSSS;SSSS;SRRRSSRSSRRSSRSS;SSSS...,6-4 6-4,0,57
6,2228887,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Stanislas Wawrinka,Peter Luczak,1,SSSS;RRSSSS;SRSSRS;RRSSSS;SSSRS;SSSRRS;RSSSS;R...,6-3 7-5,0,82
7,2229260,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Frederico Gil,Fernando Verdasco,2,SSRSS;SSSS;SRSRRSRSRSSRRR;SSSRRS;SRSSRS;SRSRRS...,6-3 6-2,0,144
8,2229257,29 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Jarkko Nieminen,Nicolas Almagro,2,SSSRRS;SSSS;SRRRSSSS;SSSRS;RSSRSS;SSRRSS;SRSSR...,7-6(3) 6-3,0,1055
9,2234483,29 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Andreas Haider-Maurer,Mikhail Youzhny,2,RRRSSR;SSSS;SRSSS;RSSSS;RRSSSS;SSRRRR;RRSSRR;R...,6-4 5-7 6-4,0,139


## 3. Score Validation
### Purpose
Verify that the match winner in the data matches the winner determined by the match score.

### Function: `score_winner_check()`
Counts sets won by each player based on the score string and compares to recorded winner.

In [ ]:


def score_winner_check(row):
    sets_p1 = 0
    sets_p2 = 0

    for set_score in row['score'].split():
        if '(' in set_score:  # remove tiebreak info
            set_score = set_score.split('(')[0]
        g1, g2 = map(int, set_score.split('-'))
        if g1 > g2:
            sets_p1 += 1
        else:
            sets_p2 += 1

    predicted_winner = 1 if sets_p1 > sets_p2 else 2
    return predicted_winner == row['winner']

df['score_winner_valid'] = df.apply(score_winner_check, axis=1)

df['score_winner_valid'].value_counts()


score_winner_valid
True     5749
False    5153
Name: count, dtype: int64

## 4. Point-by-Point Data Cleaning
### Clean PBP Format
Standardize the pbp field by removing trailing whitespace and ensuring pbp end with '.' show represent a set complete

In [47]:
df['pbp_fixed'] = df['pbp'].str.rstrip()

df['pbp_fixed'] = df['pbp_fixed'].apply(
    lambda x: x if x.endswith('.') else x + '.'
)


### Verify Set Count Consistency
Check if the number of sets in pbp data matches the score string.

In [48]:
df['pbp_set_count'] = df['pbp_fixed'].str.count(r'\.')
df['score_set_count'] = df['score'].str.count(r'\d-\d')

df['set_count_match'] = df['pbp_set_count'] == df['score_set_count']

df['set_count_match'].value_counts()


set_count_match
True    10902
Name: count, dtype: int64

## 5. Point Classification
### Function: `normalize_point()`
Convert point notation codes to standard format:
- 'A' (Ace) → 'S' (Server wins)
- 'D' (Double fault) → 'R' (Receiver wins)

In [49]:
def normalize_point(ch):
    if ch == 'A':  # Ace
        return 'S'
    if ch == 'D':  # Double fault
        return 'R'
    return ch


## 6. Game Winner Determination
### Function: `get_game_winner()`
Parse individual games from pbp data and determine winner by counting point wins.

**Parameters:**
- `game`: String like 'SASR/DSSR' (S=server wins, R=receiver wins)
- `starting_server`: Which player serves (1 or 2)

**Returns:** Winner (1 or 2)

In [50]:
def get_game_winner(game, starting_server):
    """
    game: string like 'SASR/DSSR'
    starting_server: 1 or 2
    """
    server = starting_server
    p1_points = 0
    p2_points = 0

    for ch in game:
        if ch == '/':
            # explicit serve switch inside tie-break
            server = 2 if server == 1 else 1
            continue

        ch = normalize_point(ch)

        if ch == 'S':
            winner = server
        elif ch == 'R':
            winner = 2 if server == 1 else 1
        else:
            continue

        if winner == 1:
            p1_points += 1
        else:
            p2_points += 1

    return 1 if p1_points > p2_points else 2


## 7. Set Winner Determination
### Function: `pbp_set_winners()`
Extract all sets from pbp data and determine set winners by counting games won.

**Logic:**
- Splits pbp string by '.' (sets) and ';' (games)
- Tracks server rotation after each game
- Returns list of set winners

In [51]:
def pbp_set_winners(pbp_fixed):
    sets = [s for s in pbp_fixed.split('.') if s.strip()]
    server = 1  # server1 serves first set
    set_winners = []

    for set_data in sets:
        games = [g for g in set_data.split(';') if g.strip()]
        p1_games, p2_games = 0, 0

        for game in games:
            game_winner = get_game_winner(game, server)

            if game_winner == 1:
                p1_games += 1
            else:
                p2_games += 1

            # server switches after each game
            server = 2 if server == 1 else 1

        set_winners.append(1 if p1_games > p2_games else 2)

    return set_winners


In [52]:
df.head(10)

,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,adf_flag,wh_minutes,score_winner_valid,pbp_fixed,pbp_set_count,score_set_count,set_count_match
0,2231275,28 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Olivier Rochus,Fabio Fognini,2,SSSS;RRRR;SSRRSS;SSRRSS;RSRSRSRR;SSRSS;RSRRSR;...,6-4 6-1,0,66,False,SSSS;RRRR;SSRRSS;SSRRSS;RSRSRSRR;SSRSS;RSRRSR;...,2,2,True
1,2231276,28 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Robin Haase,Marin Cilic,2,SSRSS;RRSSRSSS;SSSS;RSSSS;SRSRSS;RSRSRSSS;RSRS...,4-6 6-4 6-3,0,141,False,SSRSS;RRSSRSSS;SSSS;RSSSS;SRSRSS;RSRSRSSS;RSRS...,3,3,True
2,2236280,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Marin Cilic,Andreas Seppi,1,SSSS;SRRRR;SSRRRSSS;RSRRSSSS;RSRSSS;SRRRR;SSRS...,6-1 6-3,0,71,True,SSSS;SRRRR;SSRRRSSS;RSRRSSSS;RSRSSS;SRRRR;SSRS...,2,2,True
3,2233792,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Alexandr Dolgopolov,Albert Ramos,1,SRSSRS;RRSSSRSS;RSRSSRSRSS;RRRR;SSRSS;SRSSS;SR...,6-3 7-5,0,83,True,SRSSRS;RRSSSRSS;RSRSSRSRSS;RRRR;SSRSS;SRSSS;SR...,2,2,True
4,2233791,29 Jul 11,ATPStudenaCroatiaOpen-ATPUmag2011,ATP,Main,Carlos Berlocq,Juan Carlos Ferrero,2,RRRR;SRSSS;SSRRSS;RSRSSS;RRRSR;SSSRRS;RRRR.SRR...,6-1 7-6(5),0,150,False,RRRR;SRSSS;SSRRSS;RSRSSS;RRRSR;SSSRRS;RRRR.SRR...,2,2,True
5,2229262,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Matthias Bachinger,Julien Benneteau,2,SSSS;SSSS;RRRR;SSSS;SSSS;SRRRSSRSSRRSSRSS;SSSS...,6-4 6-4,0,57,False,SSSS;SSSS;RRRR;SSSS;SSSS;SRRRSSRSSRRSSRSS;SSSS...,2,2,True
6,2228887,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Stanislas Wawrinka,Peter Luczak,1,SSSS;RRSSSS;SRSSRS;RRSSSS;SSSRS;SSSRRS;RSSSS;R...,6-3 7-5,0,82,True,SSSS;RRSSSS;SRSSRS;RRSSSS;SSSRS;SSSRRS;RSSSS;R...,2,2,True
7,2229260,28 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Frederico Gil,Fernando Verdasco,2,SSRSS;SSSS;SRSRRSRSRSSRRR;SSSRRS;SRSSRS;SRSRRS...,6-3 6-2,0,144,False,SSRSS;SSSS;SRSRRSRSRSSRRR;SSSRRS;SRSSRS;SRSRRS...,2,2,True
8,2229257,29 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Jarkko Nieminen,Nicolas Almagro,2,SSSRRS;SSSS;SRRRSSSS;SSSRS;RSSRSS;SSRRSS;SRSSR...,7-6(3) 6-3,0,1055,False,SSSRRS;SSSS;SRRRSSSS;SSSRS;RSSRSS;SSRRSS;SRSSR...,2,2,True
9,2234483,29 Jul 11,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Andreas Haider-Maurer,Mikhail Youzhny,2,RRRSSR;SSSS;SRSSS;RSSSS;RRSSSS;SSRRRR;RRSSRR;R...,6-4 5-7 6-4,0,139,False,RRRSSR;SSSS;SRSSS;RSSSS;RRSSSS;SSRRRR;RRSSRR;R...,3,3,True


## 8. PBP Data Validation
Apply `pbp_set_winners()` to extract set patterns and verify.

In [ ]:
df['pbp_set_winners'] = df['pbp_fixed'].apply(pbp_set_winners)
df['set_match'] = df['pbp_set_winners'] == df['score_winner_valid']



## 9. Match Winner from Sets
### Function: `match_winner_from_sets()`
Determine overall match winner by counting set wins.

In [54]:
def match_winner_from_sets(sets):
    return 1 if sets.count(1) > sets.count(2) else 2

df['pbp_match_winner'] = df['pbp_set_winners'].apply(match_winner_from_sets)
df['winner_match'] = df['pbp_match_winner'] == df['winner']

# df['winner_match'].value_counts(normalize=True)


## 10. Score String Parsing
### Function: `get_set_winners_from_score()`
Extract set winners directly from score string (e.g., "6-4 3-6 7-5").

**Handles:**
- Tiebreak notation: '7-6(5)' → '7-6'
- Retired matches: 'RET'

In [ ]:
def get_set_winners_from_score(score_string):
    set_winners = []
    # Split the score string into individual set scores
    individual_set_scores = score_string.split()

    for set_score in individual_set_scores:
        # Remove tie-break information if present (e.g., '7-6(5)' -> '7-6')
        if '(' in set_score:
            set_score = set_score.split('(')[0]
        
        # Split games for player 1 and player 2
        try:
            g1, g2 = map(int, set_score.split('-'))
            if g1 > g2:
                set_winners.append(1) # Player 1 won this set
            elif g2 > g1:
                set_winners.append(2) # Player 2 won this set
           
        except ValueError: 
            pass

    return set_winners


In [56]:
df['score_set_winners'] = df['score'].apply(get_set_winners_from_score)

display(df[['score', 'score_set_winners', 'pbp_set_winners']].head())

,score,score_set_winners,pbp_set_winners
0,6-4 6-1,"[1, 1]","[2, 2]"
1,4-6 6-4 6-3,"[2, 1, 1]","[1, 2, 2]"
2,6-1 6-3,"[1, 1]","[1, 1]"
3,6-3 7-5,"[1, 1]","[1, 1]"
4,6-1 7-6(5),"[1, 1]","[2, 2]"


In [57]:
df['set_match'] = df['pbp_set_winners'] == df['score_set_winners']

# Let's see the distribution of this corrected set_match column
df['set_match'].value_counts(normalize=True)

set_match
True     0.527243
False    0.472757
Name: proportion, dtype: float64

In [58]:
df[df['pbp_id'] == '2306737']


,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,...,score_winner_valid,pbp_fixed,pbp_set_count,score_set_count,set_count_match,pbp_set_winners,set_match,pbp_match_winner,winner_match,score_set_winners


In [59]:
df.sample(10)

,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,...,score_winner_valid,pbp_fixed,pbp_set_count,score_set_count,set_count_match,pbp_set_winners,set_match,pbp_match_winner,winner_match,score_set_winners
4184,4331506,18 Apr 13,Monte-CarloRolexMasters-ATPMonaco.html,ATP,Main,Jo-Wilfried Tsonga,Jurgen Melzer,1,RSRASS;SRSSRS;ASSRS;RSRRR;SAARRS;RRSSSRSS;SSSS...,6-3 6-0,...,True,RSRASS;SRSSRS;ASSRS;RSRRR;SAARRS;RRSSSRSS;SSSS...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
3225,3896998,31 Dec 12,AircelChennaiOpen-ATPChennai,ATP,Main,Kenny De Schepper,Rajeev Ram,2,SSSS;SSSRS;RSRRSSRSRSSS;RRSRSR;RRRR;SSSRS;SRRR...,7-6(7) 4-6 6-1,...,False,SSSS;SSSRS;RSRRSSRSRSSS;RRSRSR;RRRR;SSSRS;SRRR...,3,3,True,"[2, 1, 2]",False,2,True,"[1, 2, 1]"
1248,2885044,13 Apr 12,GrandPrixHassanII-ATPCasablanca2012,ATP,Main,Sergio Gutierrez-Ferrol,Pablo Andujar,2,SRRRR;SSSRS;RRRR;SSRRRR;SSSS;SSSS;RSSSRRRSRR;S...,6-4 1-6 6-3,...,False,SRRRR;SSSRS;RRRR;SSRRRR;SSSS;SSSS;RSSSRRRSRR;S...,3,3,True,"[2, 1, 2]",False,2,True,"[1, 2, 1]"
6063,5474885,13 Jan 14,Men'sAustralianOpen.,ATP,Main,Ivo Karlovic,Ivan Dodig,2,SRSRSS;RSARSS;SSAA;RSRSRR;SSSRS;SSSS;SARRAS;DS...,7-6(8) 6-3 7-6(4),...,False,SRSRSS;RSARSS;SSAA;RSRSRR;SSSRS;SSSS;SARRAS;DS...,3,3,True,"[2, 2, 2]",False,2,True,"[1, 1, 1]"
10362,8097393,01 Sep 15,Men'sUSOpen,ATP,Main,Paolo Lorenzi,Jiri Vesely,2,RSSSRS;AASRS;SSSRA;RSRARR;SRSRRR;DSSRSS;SSSA;A...,6-4 6-4 6-4,...,False,RSSSRS;AASRS;SSSRA;RSRARR;SRSRRR;DSSRSS;SSSA;A...,3,3,True,"[2, 2, 2]",False,2,True,"[1, 1, 1]"
3727,4080019,16 Feb 13,ABNAMROWorldTennisTournament-ATPRotterdam,ATP,Main,Gilles Simon,Julien Benneteau,2,RRASRSAS;RRSSSRSS;SSARS;SSRSS;SSSS;SRSSRS;RSSS...,6-4 7-6(2),...,False,RRASRSAS;RRSSSRSS;SSARS;SSRSS;SSSS;SRSSRS;RSSS...,2,2,True,"[2, 2]",False,2,True,"[1, 1]"
3901,4128830,28 Feb 13,AbiertoMexicanoTelcel-ATPAcapulco,ATP,Main,Horacio Zeballos,Daniel Gimeno-Traver,1,SSSS;RSRRAD;DRRSR;RSRRSR;SRRARSSS;SAADS;SSSRS;...,6-2 6-3,...,True,SSSS;RSRRAD;DRRSR;RSRRSR;SRRARSSS;SAADS;SSSRS;...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
6123,5577590,04 Feb 14,PBZZagrebIndoors-ATPZagreb,ATP,Main,Michael Berrer,Borna Coric,1,SSSA;DSARRSSS;SRRSSS;RASRARRR;SSSS;RSSRSS;SRSR...,6-4 3-6 6-4,...,True,SSSA;DSARRSSS;SRRSSS;RASRARRR;SSSS;RSSRSS;SRSR...,3,3,True,"[1, 2, 1]",True,1,True,"[1, 2, 1]"
8391,7043840,07 Jan 15,QatarExxonMobilOpen-ATPQatar,ATP,Main,Tomas Berdych,Blaz Kavcic,1,SSRSRS;SRSRRR;SRSSS;SADSRRRR;AARRSS;SRSRSS;SSS...,6-1 6-2,...,True,SSRSRS;SRSRRR;SRSSS;SADSRRRR;AARRSS;SRSRSS;SSS...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
1477,2977559,14 May 12,InternazionaliBNLd'Italia-ATPRome,ATP,Main,Nicolas Almagro,Marin Cilic,1,SSRSRS;RRSRSR;SSSRRRRSSS;SSSS;SRSRSS;SSSS;SRSR...,6-2 3-6 6-0,...,True,SSRSRS;RRSRSR;SSSRRRRSSS;SSSS;SRSRSS;SSSS;SRSR...,3,3,True,"[1, 2, 1]",True,1,True,"[1, 2, 1]"


### Missing Values Analysis

In [60]:
df.isna().sum().sort_values(ascending=False)


pbp_id                0
date                  0
winner_match          0
pbp_match_winner      0
set_match             0
pbp_set_winners       0
set_count_match       0
score_set_count       0
pbp_set_count         0
pbp_fixed             0
score_winner_valid    0
wh_minutes            0
adf_flag              0
score                 0
pbp                   0
winner                0
server2               0
server1               0
draw                  0
tour                  0
tny_name              0
score_set_winners     0
dtype: int64

### Duplicate Detection

In [61]:
df['pbp_id'].nunique()



10828

In [62]:
df[df.duplicated(subset='pbp_id', keep=False)]


,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,...,score_winner_valid,pbp_fixed,pbp_set_count,score_set_count,set_count_match,pbp_set_winners,set_match,pbp_match_winner,winner_match,score_set_winners
19,2239568,01 Aug 11,ATPKitzbuhel2011,ATP,Main,Pere Riba,Victor Hanescu,1,SRRRSSSRSRSS;SRRRSR;SSSRS;SSSS;RRSRR;SSRRSRSS;...,7-6(3) 6-4,...,True,SRRRSSSRSRSS;SRRRSR;SSSRS;SSSS;RRSRR;SSRRSRSS;...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
29,2244796,02 Aug 11,ATPKitzbuhel2011,ATP,Main,Daniel Brands,Santiago Giraldo,2,RSSRRR;SSSS;SSSS;SSSRS;SSSS;SSSRS;SSSS;SSSS;SR...,6-3 6-3,...,False,RSSRRR;SSSS;SSSS;SSSRS;SSSS;SSSRS;SSSS;SSSS;SR...,2,2,True,"[2, 2]",False,2,True,"[1, 1]"
34,2239568,01 Aug 11,ATPKitzbuhel2011.html,ATP,Main,Pere Riba,Victor Hanescu,1,SRRRSSSRSRSS;SRRRSR;SSSRS;SSSS;RRSRR;SSRRSRSS;...,7-6(3) 6-4,...,True,SRRRSSSRSRSS;SRRRSR;SSSRS;SSSS;RRSRR;SSRRSRSS;...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
35,2244796,02 Aug 11,ATPKitzbuhel2011.html,ATP,Main,Daniel Brands,Santiago Giraldo,2,RSSRRR;SSSS;SSSS;SSSRS;SSSS;SSSRS;SSSS;SSSS;SR...,6-3 6-3,...,False,RSSRRR;SSSS;SSSS;SSSRS;SSSS;SSSRS;SSSS;SSSS;SR...,2,2,True,"[2, 2]",False,2,True,"[1, 1]"
58,2239498,02 Aug 11,LeggMasonTennisClassic-ATPWashington2011,ATP,Main,Ryan Harrison,Mischa Zverev,1,SSSS;RSSRSS;SSSS;RRRSSSSRSS;SSSRRRSS;SSRSRS;SS...,6-4 1-6 6-1,...,True,SSSS;RSSRSS;SSSS;RRRSSSSRSS;SSSRRRSS;SSRSRS;SS...,3,3,True,"[1, 2, 1]",True,1,True,"[1, 2, 1]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5204,4805825,09 Aug 13,RogersCup-ATPMontreal.html,ATP,Main,Ernests Gulbis,Milos Raonic,2,SSRSRS;DASDSS;SRSSA;SASRS;DRSSSDAS;ASSS;SSSDRS...,7-6(3) 4-6 6-4,...,False,SSRSRS;DASDSS;SRSSA;SASRS;DRSSSDAS;ASSS;SSSDRS...,3,3,True,"[2, 1, 2]",False,2,True,"[1, 2, 1]"
5205,4805812,09 Aug 13,RogersCup-ATPMontreal.html,ATP,Main,Novak Djokovic,Richard Gasquet,1,SASA;RSRSDR;RRSSRSSRRASS;SDSSRA;SSSS;RRSRR;SSA...,6-1 6-2,...,True,SASA;RSRSDR;RRSSRSSRRASS;SDSSRA;SSSS;RRSRR;SSA...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
5209,4805506,09 Aug 13,RogersCup-ATPMontreal,ATP,Main,Denis Istomin,Novak Djokovic,2,SSSS;ASSS;RSRSSS;SSRRSS;SRSAS;RSRSRSSRARSDRD;S...,2-6 6-4 6-4,...,False,SSSS;ASSS;RSRSSS;SSRRSS;SRSAS;RSRSRSSRARSDRD;S...,3,3,True,"[1, 2, 2]",False,2,True,"[2, 1, 1]"
5210,4805825,09 Aug 13,RogersCup-ATPMontreal,ATP,Main,Ernests Gulbis,Milos Raonic,2,SSRSRS;DASDSS;SRSSA;SASRS;DRSSSDAS;ASSS;SSSDRS...,7-6(3) 4-6 6-4,...,False,SSRSRS;DASDSS;SRSSA;SASRS;DRSSSDAS;ASSS;SSSDRS...,3,3,True,"[2, 1, 2]",False,2,True,"[1, 2, 1]"


In [63]:
df.describe()

,pbp_id,winner,adf_flag,wh_minutes,pbp_set_count,score_set_count,pbp_match_winner
count,1.090200e+04,10902.000000,10902.000000,10902.000000,10902.000000,10902.000000,10902.000000
mean,5.167553e+06,1.472757,0.653183,43.469363,2.590809,2.590809,1.472849
std,1.839888e+06,0.499280,0.475979,283.547448,0.743969,0.743969,0.499285
min,2.228887e+06,1.000000,0.000000,-1398.000000,2.000000,2.000000,1.000000
25%,3.489634e+06,1.000000,0.000000,69.000000,2.000000,2.000000,1.000000
50%,4.973090e+06,1.000000,1.000000,91.000000,2.000000,2.000000,1.000000
75%,6.682608e+06,2.000000,1.000000,122.000000,3.000000,3.000000,2.000000
max,8.634739e+06,2.000000,1.000000,1405.000000,5.000000,5.000000,2.000000


In [64]:
df['pbp_id'].value_counts()[df['pbp_id'].value_counts() > 1]


pbp_id
2306737    4
2311567    4
2306736    4
2282506    3
2321684    3
2319098    3
2319093    3
2326600    3
2326596    3
2329512    3
2332231    3
2306758    3
2332233    3
2334439    3
2334441    3
2340460    3
2319094    3
2280151    3
2276269    3
2492442    2
2244641    2
2282503    2
2290527    2
2466656    2
2257491    2
2257490    2
2257487    2
4805506    2
2257499    2
2303077    2
2459258    2
4805825    2
4805812    2
2495348    2
2495349    2
2495351    2
2492445    2
2492444    2
2492441    2
2239498    2
2438631    2
2360867    2
2499834    2
2244796    2
2440956    2
2501980    2
2440988    2
2495353    2
2497248    2
2497239    2
2422754    2
2239568    2
Name: count, dtype: int64

## 11. Data Cleaning
### Remove Duplicates

In [65]:
df = df.drop_duplicates(subset='pbp_id')


In [66]:
df = df[df['pbp_set_count'] <= 3]


### Clean Duration and Dates
- Duration: 30–500 minutes
- Parse dates with coercion
- Drop rows missing critical fields

In [67]:
# 1. Fix duration errors
df = df[(df['wh_minutes'] > 30) & (df['wh_minutes'] < 500)].copy()

# 2. Fix date formats
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 3. Drop rows with missing critical info
df = df.dropna(subset=['date', 'pbp_set_winners'])

/var/folders/rz/3q9t0_5s4tjdszvw2wf1swvh0000gn/T/ipykernel_34554/3277969807.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [68]:
df.sample(10)

,pbp_id,date,tny_name,tour,draw,server1,server2,winner,pbp,score,...,score_winner_valid,pbp_fixed,pbp_set_count,score_set_count,set_count_match,pbp_set_winners,set_match,pbp_match_winner,winner_match,score_set_winners
5976,5435561,2014-01-01,QatarExxonMobilOpen-ATPQatar2014,ATP,Main,Florian Mayer,Andy Murray,1,SRSSRS;SRSSS;SRRRSSAS;SSSS;SRRSRR;RDSASRRSAA;S...,3-6 6-4 6-2,...,True,SRSSRS;SRSSS;SRRRSSAS;SSSS;SRRSRR;RDSASRRSAA;S...,3,3,True,"[2, 1, 1]",True,1,True,"[2, 1, 1]"
6,2228887,2011-07-28,CreditAgricoleSuisseOpenGstaad-ATPGstaad2011,ATP,Main,Stanislas Wawrinka,Peter Luczak,1,SSSS;RRSSSS;SRSSRS;RRSSSS;SSSRS;SSSRRS;RSSSS;R...,6-3 7-5,...,True,SSSS;RRSSSS;SRSSRS;RRSSSS;SSSRS;SSSRRS;RSSSS;R...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
8425,7077072,2015-01-15,ApiaInternationalSydney-ATPSydney,ATP,Main,Leonardo Mayer,Julien Benneteau,1,SRDRSSRSAA;DRSSRR;SRSRSRSS;RRSSSS;DSSSA;RSSRSR...,6-3 7-6(4),...,True,SRDRSSRSAA;DRSSRR;SRSRSRSS;RRSSSS;DSSSA;RSSRSR...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
2290,3226316,2012-07-16,GermanTennisChampionships2012-ATPHamburg,ATP,Main,Federico Delbonis,Tommy Robredo,1,RRSRSR;SSSS;SSSRS;RSSRRR;SSRSRS;SSSRS;RSSSS;SS...,5-7 6-0 6-1,...,True,RRSRSR;SSSS;SSSRS;RSSRRR;SSRSRS;SSSRS;RSSSS;SS...,3,3,True,"[2, 1, 1]",True,1,True,"[2, 1, 1]"
10518,8198360,2015-09-18,DavisCup,ATP,Main,Federico Delbonis,David Goffin,2,RSRSRSDASS;SSSS;SRSAS;SSSRS;RSRSRSRSSRRSDR;SSA...,7-5 7-6(3) 6-3,...,False,RSRSRSDASS;SSSS;SRSAS;SSSRS;RSRSRSRSSRRSDR;SSA...,3,3,True,"[2, 2, 2]",False,2,True,"[1, 1, 1]"
144,2293289,2011-08-23,WinstonSalemOpen-ATPWinstonSalem2011,ATP,Main,Lleyton Hewitt,Blaz Kavcic,2,SSRRSRSRRR;RRRR;RRSSRSRR;SRSSS;SSRRSS;SRRSSRRS...,6-4 7-6(3),...,False,SSRRSRSRRR;RRRR;RRSSRSRR;SRSSS;SSRRSS;SRRSSRRS...,2,2,True,"[2, 2]",False,2,True,"[1, 1]"
2560,3383353,2012-08-28,Men'sUSOpen,ATP,Main,Roger Federer,Donald Young,1,RSSSS;SRRRSSSS;SSSS;SRSSRRSS;SRSSS;SSSRS;SRSSR...,6-3 6-2 6-4,...,True,RSSSS;SRRRSSSS;SSSS;SRSSRRSS;SRSSS;SSSRS;SRSSR...,3,3,True,"[1, 1, 1]",True,1,True,"[1, 1, 1]"
8796,7181409,2015-02-07,PBZZagrebIndoors-ATPZagreb,ATP,Main,Andreas Seppi,Marcel Granollers,1,SSRRSS;RSSRSS;SRARSRRR;SSRRRR;SSSRS;SRRSDSARSS...,7-6(5) 6-1,...,True,SSRRSS;RSSRSS;SRARSRRR;SSRRRR;SSSRS;SRRSDSARSS...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
4128,4219870,2013-03-23,SonyOpenTennis-ATPMiami,ATP,Main,Novak Djokovic,Lukas Rosol,1,RASSS;SRSSRRRR;SSSS;RSSSA;SASS;RRSDR;SRSRRSRSS...,6-1 6-0,...,True,RASSS;SRSSRRRR;SSSS;RSSSA;SASS;RRSDR;SRSRRSRSS...,2,2,True,"[1, 1]",True,1,True,"[1, 1]"
9450,7557960,2015-05-02,TEBBNPParibasIstanbulOpen-ATPIstanbul,ATP,Main,Roger Federer,Diego Schwartzman,1,SSRSS;RRSSSS;RSRSRSRR;SRRSSRSRSS;SRSSS;SSSRS;R...,2-6 6-2 7-5,...,True,SSRSS;RRSSSS;RSRSRSRR;SRRSSRSRSS;SRSSS;SSSRS;R...,3,3,True,"[2, 1, 1]",True,1,True,"[2, 1, 1]"


## 12. Final Feature Removal
Drop non-essential columns for downstream analysis.

In [69]:
df = df.drop('tny_name', axis=1) 
df = df.drop('tour', axis=1) 
df = df.drop('draw', axis=1) 
df = df.drop('adf_flag', axis=1) 


## 13. Export Cleaned Dataset

In [70]:
df.to_csv('Cleaned_dataset.csv', index=False)
